# Module 2: Web search with Tavily - your first agentic loop

Large language models are frozen in time — they only know what was in their training data. To answer questions about *what's happening now* (say, this week's AI news), we have to **give the model a tool** and let it go fetch fresh information.

In this notebook we build that from scratch, with no agent framework:

1. Meet the two Tavily tools we'll use: **Search** (find pages) and **Extract** (read pages in full).
2. Call both APIs directly over HTTP, then via the Python client.
3. Turn them into **tools** — plain Python functions the model can call.
4. Wire a **ReAct-style agentic loop** in ~20 lines: the model reasons, calls search/extract, reads the results, and repeats until it can answer.

By the end you'll have a tiny research agent that answers a question with real, cited sources — and you'll understand every line of it.


## 0 — Get your API keys

You need two keys, both placed in a `.env` file inside this `lesson/` folder. Copy over the `.env.example` to a new file, rename it to `.env`.

**Tavily** (web search)
1. Go to [tavily.com](https://www.tavily.com) and sign in with Google or GitHub.
2. Copy an API key from the dashboard — it looks like `tvly-...`.
3. Free tier: **1,000 credits/month, no credit card**. A basic search costs 1 credit.

**Nebius Token Factory** (the LLM)
1. Go to [tokenfactory.nebius.com](https://tokenfactory.nebius.com) and create an account.
2. Open **API keys → Create API key** and copy it (you can't view it again later).
3. Nebius exposes an **OpenAI-compatible** API, so we drive it with the plain `openai` SDK.

Your `lesson/.env` should look like:

```
TAVILY_API_KEY=tvly-...
NEBIUS_API_KEY=...
```

In [ ]:
import os

from dotenv import load_dotenv
from rich import print

load_dotenv()
%load_ext rich

assert os.environ.get("TAVILY_API_KEY"), "Missing TAVILY_API_KEY in lesson/.env"
assert os.environ.get("NEBIUS_API_KEY"), "Missing NEBIUS_API_KEY in lesson/.env"
print("Keys loaded.")

## The two tools we'll use today

Tavily is a search API built for LLMs — results come back as clean, structured JSON instead of raw HTML. Today we focus on two endpoints that cover most agent research workflows:

| Tool | What it does | When you use it |
|------|--------------|-----------------|
| **Tavily Search** | Given a query, finds the most relevant pages on the web and returns ranked results with titles, URLs, and content snippets. | You don't know the URL yet — you need to *discover* sources. |
| **Tavily Extract** | Given one or more URLs you already have, pulls the full page content as clean markdown/text. | You already know *where* to look — you need the page itself, not just a snippet. |

Think of them as a two-step research pattern:

1. **Search** to find promising sources.
2. **Extract** to read the ones that matter in full.

Search alone is often enough for a quick answer. Extract shines when snippets aren't enough — long docs, pricing pages, blog posts, or anything where the model needs the full text to reason accurately.

We'll call both over HTTP, wrap them as tools, then let the model decide when to search vs. extract.


## 1 — Call Tavily directly over HTTP

Before any client library or agent framework, we will see what Tavily *actually is*. At its core, each capability is a single HTTP endpoint you POST to. Everything else is a wrapper around that.

We'll call both endpoints by hand: **Search** first, then **Extract** on a URL from the results.

### Search

- **Endpoint:** `POST https://api.tavily.com/search`
- **Auth:** an `Authorization: Bearer <your-key>` header
- **Body:** JSON with your `query` plus a few optional knobs

The parameters worth knowing:

| Param | What it does |
|-------|--------------|
| `query` | The search string (required) |
| `search_depth` | `basic`/`fast` (1 credit) or `advanced` (2 credits, deeper) |
| `max_results` | How many results to return (1–20) |
| `chunks_per_source` | How many snippets to return per source (1–3) |
| `include_answer` | Also return a one-shot LLM answer to the query |
| `time_range` | `day` / `week` / `month` / `year` to filter by recency |
| `include_raw_content` | Return the raw HTML content of the page |


In [ ]:
import requests

response = requests.post(
    "https://api.tavily.com/search",
    headers={"Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}"},
    json={
        "query": "advancements in AI this week",
        "search_depth": "advanced",
        "max_results": 5,
        "time_range": "week",
        "include_answer": True,
    },
    timeout=30,
)
response.raise_for_status()
data = response.json()

print(f"Got {len(data['results'])} results in {data['response_time']}s")
print(list(data.keys()))

### What comes back

The response is a JSON object. The parts we care about:

- **`results`** — a ranked list; each item has `title`, `url`, `content` (a relevant text snippet), and a `score` (relevance, 0–1).
- **`answer`** — Tavily's own one-line answer (present only because we set `include_answer`).
- **`response_time`**, **`request_id`** — metadata.

`content` is the key field: short, relevant chunks we can feed straight to an LLM without scraping full pages.

In [ ]:
print("Tavily's answer:", data.get("answer"))

print("Results: ")
for i, r in enumerate(data["results"]):
    print(f"""({i + 1}) [{r["score"]:.2f}] {r["title"]}
{r["url"]}
{r["content"][:160]}...""")

### Extract

Search returned ranked snippets. When a snippet isn't enough, we pass the URL(s) we care about to **Extract** and get the full page as clean markdown.

- **Endpoint:** `POST https://api.tavily.com/extract`
- **Auth:** same Bearer token
- **Body:** JSON with `urls` (required) plus optional knobs

| Param | What it does |
|-------|--------------|
| `urls` | One URL or a list (max 20) |
| `extract_depth` | `basic` (faster) or `advanced` (better on JS-heavy pages) |
| `format` | `markdown` (default) or `text` |
| `query` | Optional focus string — reranks chunks toward what's relevant |
| `chunks_per_source` | With `query`, how many chunks per page (1–5) |

Below we extract the top result from the search we just ran — the classic search → extract pattern.

In [ ]:
# Reuse the last URL from the search response above.
url_to_extract = data["results"][-1]["url"]
print(f"Extracting: {url_to_extract}")

extract_response = requests.post(
    "https://api.tavily.com/extract",
    headers={"Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}"},
    json={
        "urls": [url_to_extract],
        "extract_depth": "advanced",
        "format": "markdown",
    },
    timeout=60,
)
extract_response.raise_for_status()
extract_data = extract_response.json()

print(f"Got {len(extract_data['results'])} page(s) in {extract_data.get('response_time')}s")
print(list(extract_data.keys()))

### What comes back from Extract

- **`results`** — one entry per successful URL, each with `url` and `raw_content` (the cleaned page text/markdown).
- **`failed_results`** — URLs that couldn't be fetched, with an error reason.
- **`response_time`**, **`request_id`** — metadata.

Compare `raw_content` to the short `content` snippet from Search — this is the full article the model can actually read.


In [ ]:
for r in extract_data["results"]:
    content = r["raw_content"] or ""
    print(f"URL: {r['url']}")
    print(f"Length: {len(content)} chars")
    print(f"Preview:\n{content[:500]}\n\n[red]***[TRUNCATED FOR NOTEBOOK]***[/red]\n\n{content[-500:]}")

if extract_data.get("failed_results"):
    print("Failed:", extract_data["failed_results"])

## 2 — The same thing with the Python client

Writing the HTTP call by hand is great for understanding, but tedious in practice. The `tavily-python` client wraps those POST requests — same parameters, less boilerplate. This is what we'll use for the rest of the notebook (both `search` and `extract`).

In [ ]:
from tavily import TavilyClient

# The `client_name` is optional, but it helps track attribution
tavily_client = TavilyClient(
    api_key=os.environ["TAVILY_API_KEY"], client_name="nv-course-ai-agents"
)

data = tavily_client.search(
    "advancements in AI this week",
    search_depth="advanced",
    max_results=5,
    time_range="week",
)

print(f"{len(data['results'])} results")

In [ ]:
print("Results: ")
for i, r in enumerate(data["results"]):
    print(f"""({i + 1}) [{r["score"]:.2f}] {r["title"]}
{r["url"]}
{r["content"][:160]}...""")

## 3 — A "tool" is just a function

An agent tool is nothing exotic — it's a normal Python function with a clear input and output. Here we wrap both Tavily endpoints:

- `internet_search()` — discover sources from a query
- `extract_content()` — read full pages from URLs we already have

We also add a small helper that flattens search results into a compact string we can hand back to the model.


In [ ]:
def format_results(results: list[dict]) -> str:
    """Turn search results into a compact, model-friendly string."""
    if not results:
        return "No results found."
    return "\n\n".join(
        f"({i + 1}) Title: {r['title']}\nURL: {r['url']}\nContent: {r['content']}"
        for i, r in enumerate(results)
    )


def internet_search(
    query: str, search_depth: str = "advanced", max_results: int = 5
) -> str:
    """Search the web and return a formatted string of {title, url, content} results."""
    data = tavily_client.search(
        query, search_depth=search_depth, max_results=max_results
    )
    return format_results(data["results"])


def extract_content(urls: list[str], query: str | None = None) -> str:
    """Extract full page content from one or more URLs as markdown."""
    kwargs: dict = {
        "urls": urls,
        "extract_depth": "advanced",
        "format": "markdown",
    }
    if query:
        kwargs["query"] = query

    data = tavily_client.extract(**kwargs)
    results = data.get("results") or []
    if not results:
        failed = data.get("failed_results") or []
        return (
            f"No content extracted. Failures: {failed}"
            if failed
            else "No content extracted."
        )

    parts = []
    for i, r in enumerate(results):
        content = r.get("raw_content") or ""
        parts.append(f"({i + 1}) URL: {r['url']}\nContent:\n{content}")
    return "\n\n".join(parts)


print("=== Search ===")
print(
    internet_search(
        "Which teams qualified to the Round of 16 in the FIFA World Cup 2026?",
        max_results=3,
    )
)
print("\n=== Extract ===")
print(
    extract_content(
        ["https://docs.python.org/3/whatsnew/3.13.html"],
        query="key new features",
    )[:800]
)


## 4 — Let the model decide to call the tool

Up until now, we have only worked with the "tool" part of an agent.

Now the LLM. We point the `openai` SDK at Nebius's OpenAI-compatible endpoint, then **describe** our tools to the model as JSON schemas — both Search and Extract.

The crucial idea: the model never runs the functions itself. When it decides a tool is needed, it returns a **tool call** — the function name plus the arguments it wants us to run. Executing it is *our* job.


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ["NEBIUS_API_KEY"],
)

MODEL = "nvidia/nemotron-3-super-120b-a12b"

# The schema is how the model "sees" our function: name, purpose, and arguments.
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "internet_search",
        "description": "Search the web for current, factual information. Returns results with title, url, and a content snippet. Use this first to discover sources.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query."},
                "search_depth": {
                    "type": "string",
                    "enum": ["basic", "fast", "advanced"],
                    "description": "Search depth. Use 'advanced' for more detailed results.",
                },
                "max_results": {
                    "type": "integer",
                    "description": "How many results to return (1-10).",
                },
            },
            "required": ["query"],
        },
    },
}

EXTRACT_TOOL = {
    "type": "function",
    "function": {
        "name": "extract_content",
        "description": "Extract the full content of one or more web pages given their URLs. Use after search when a snippet is not enough and you need the full page.",
        "parameters": {
            "type": "object",
            "properties": {
                "urls": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of URLs to extract (max 5).",
                },
                "query": {
                    "type": "string",
                    "description": "Optional focus query to keep only the most relevant chunks from each page.",
                },
            },
            "required": ["urls"],
        },
    },
}

AGENT_TOOLS = [SEARCH_TOOL, EXTRACT_TOOL]


In [ ]:
from datetime import datetime

messages = [
    {
        "role": "system",
        "content": (
            f"You are a research assistant. Use internet_search to find sources, "
            f"then extract_content on the most promising URLs when you need more detail. "
            f"Today is {datetime.now().strftime('%Y-%m-%d')}"
        ),
    },
    {"role": "user", "content": "What were the advancements in AI this week?"},
]

response = client.chat.completions.create(
    model=MODEL, messages=messages, tools=AGENT_TOOLS
)
assistant_msg = response.choices[0].message

print("Content (usually empty when it wants a tool):", repr(assistant_msg.content))
print("Tool calls the model is requesting:")
for tc in assistant_msg.tool_calls or []:
    print(f"  {tc.function.name}({tc.function.arguments})")


Notice the model didn't answer — it handed us a **request to act**: "please run this tool with these arguments."

Running it and returning the result is *our* job. Do that, feed the result back, and the model continues — it may search again, extract a page, or finally answer. Wrap that back-and-forth in a loop and you have a basic agent.


In [ ]:
import json

# The model asked us to run a tool. Let's fulfil that request by hand, then hand
# the results back so it can finish -- ONE turn of what will become the loop.

TOOLS = {
    "internet_search": internet_search,
    "extract_content": extract_content,
}

# 1) Record the model's tool-call turn in the conversation history.
messages.append(assistant_msg.model_dump(exclude_none=True))

# 2) Run each tool call the model requested, appending each result as a "tool" message.
for tc in assistant_msg.tool_calls:
    args = json.loads(tc.function.arguments)
    print(f"Running {tc.function.name}({args})")
    result = TOOLS[tc.function.name](**args)
    print(f"Result snippet: {result[:100]} ... {result[-100:]}")
    messages.append(
        {
            "role": "tool",
            "tool_call_id": tc.id,
            "content": result,
        }
    )

# 3) Ask the model again -- now it can see the tool results and can answer (or call again).
followup = client.chat.completions.create(
    model=MODEL, messages=messages, tools=AGENT_TOOLS
)
final = followup.choices[0].message

if final.content:
    print("\nFinal answer:\n")
    print(final.content)
else:
    print("\nThe model wants another tool call:")
    for tc in final.tool_calls or []:
        print(f"  {tc.function.name}({tc.function.arguments})")
    print("...which is exactly why we wrap this in a loop next.")


## 5 — The Pythonic agentic loop

The whole pattern is a `while` loop over a growing list of messages:

1. Ask the model, offering it both tools.
2. If it returns tool calls → run each one, append the results, loop again.
3. If it returns plain text → it's done; that text is the answer.

We add a `max_steps` guard so a confused model can't loop forever. That's the entire idea behind "agents" — everything fancier is an optimization on top of this loop.

#### Aside: Tracing
Tracing (what got called, with what args, what came back) is useful, but it's not the central focus here - if you are not able to understand the code, you can safely proceed to the next section.


Here, we are keeping a tiny `ToolTracer` on the side: the loop just calls `tracer.run(...)`, and the tracer owns the audit trail.

In [ ]:
from typing import Any, Callable

class ToolTracer:
    """Runs tools and records every call — keeps the agent loop free of bookkeeping.

    For students: This block sets up a tracer that can collect the tool calls and results made by the agent. 
    """

    def __init__(self, tools: dict[str, Callable[..., str]]):
        self._tools = tools
        self.events: list[dict[str, Any]] = []

    def clear(self) -> None:
        self.events.clear()

    def run(
        self, name: str, args: dict, *, step: int, verbose: bool = False
    ) -> str:
        if verbose:
            print(f"[bold cyan]Step {step}: {name}[/bold cyan] {args}")
        result = self._tools[name](**args)
        self.events.append(
            {"step": step, "tool": name, "args": args, "result": result}
        )
        if verbose:
            print(f"Result snippet: {result[:100]} ... {result[-100:]}")
        return result

    def show(self, result_chars: int = 240) -> None:
        print(f"[bold]Trace[/bold] — {len(self.events)} tool call(s), in order:\n")
        if not self.events:
            print("[dim](no tools were called)[/dim]")
            return
        for i, event in enumerate(self.events, 1):
            print(f"[cyan]{i}. step {event['step']} → {event['tool']}[/cyan]")
            print(f"   args:   {event['args']}")
            preview = event["result"][:result_chars].replace("\n", " ")
            suffix = "..." if len(event["result"]) > result_chars else ""
            print(f"   result: {preview}{suffix}\n")



In [ ]:
import json
from datetime import datetime
from typing import Any, Callable

TOOLS: dict[str, Callable[..., str]] = {
    "internet_search": internet_search,
    "extract_content": extract_content,
}

SYSTEM_PROMPT = f"""You are a research assistant. Use internet_search to discover sources, then extract_content on the most promising URLs when snippets are not enough. Base your answer only on what you find, and cite the source URL for every claim. Search a maximum of 3 times, and use extract_content only once. Today is {datetime.now().strftime("%Y-%m-%d")}"""

def run_agent(
    question: str,
    model: str = MODEL,
    max_steps: int = 10,
    verbose: bool = True,
    tracer: ToolTracer | None = None,
) -> str:
    tracer = tracer or ToolTracer(TOOLS)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=model, messages=messages, tools=AGENT_TOOLS
        )
        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            if verbose:
                print(f"[bold green]Step {step}: final answer[/bold green]")
            return msg.content

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = tracer.run(
                tc.function.name, args, step=step, verbose=verbose
            )
            messages.append(
                {"role": "tool", "tool_call_id": tc.id, "content": result}
            )

    return "Stopped: reached max_steps without a final answer."


In [ ]:
tracer = ToolTracer(TOOLS)

answer = run_agent(
    "What were the 3 biggest advancements in AI this week? "
    "Search first, then extract the full text of the 1-2 most useful articles before answering.",
    tracer=tracer,
)

print("[bold]Answer[/bold]\n")
print(answer)
print()
tracer.show()

# Dig into any single event:
# tracer.events[0]


The agent loop stayed small — it only asks the model, runs tools, and appends messages. All collectible state lives on the tracer:

- **`tracer.events`** — ordered `{step, tool, args, result}` records
- **`tracer.show()`** — pretty-print the trail after the run
- **`tracer.clear()`** — reset between runs (handy in the bonus below)

Streaming logs are for watching; the tracer is for *keeping*. Same idea scales up to real observability tools (LangSmith, Phoenix, etc.).


## 6 — Bonus: how good are different models at tool calling?

Not every model is equally reliable at *deciding* to call a tool, formatting *valid* arguments, and *stopping* when done. Because our loop is model-agnostic, we can point it at several Nebius models and eyeball the difference. Swap in any model IDs from the [Nebius playground](https://tokenfactory.nebius.com); bad names are caught by the `try/except` below.

In [ ]:
CANDIDATE_MODELS = [
    "nvidia/nemotron-3-super-120b-a12b",
    "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    "MiniMaxAI/MiniMax-M2.5",
    "zai-org/GLM-5.2",
]

question = "What is the latest stable version of Python, and when was it released?"

for model in CANDIDATE_MODELS:
    print(f"[bold]{model}[/bold]")
    try:
        answer = run_agent(
            question, model=model, max_steps=5, verbose=False
        )
        print(answer[:400])
        print(f"[dim]trace: {[e['tool'] for e in tracer.events] or '(no tools)'}[/dim]")
    except Exception as e:
        print(f"[red]failed: {type(e).__name__}: {e}[/red]")
    print("-" * 80)


## 7 — Where this breaks, and what's next

Our 20-line agent works, but it's naive:

- **No planning.** It reacts one search/extract at a time; it doesn't decompose a big question into sub-tasks.
- **Redundant calls.** Nothing stops it re-querying the same thing or re-extracting the same URL.
- **No source-quality control.** It trusts whatever ranks highest.
- **Context bloat.** Every result (especially full-page extracts) is stuffed back into the message list; long runs blow past the context window.

Fixing these is the job of a real **search pipeline** — query planning, result deduplication, source filtering, and summarization — which we build in the next chapter. From there we graduate to a full multi-agent **competitor-research agent** (see the project code in the repo root) that plans, delegates to parallel sub-agents, and fact-checks its own output.
